# 🚀 Lab 17: Extract Patterns Using Regular Expressions

### 📘 Lab Overview
In this lab, you will learn how to use **regular expressions (regex)** to extract structured information from messy text data. Real-world text fields often contain inconsistent formatting, mixed separators, and multiple pieces of information inside one string. Regex helps us detect patterns in that text so we can clean and standardize it.

You will work with healthcare-style records and use pandas string methods together with Python’s `re` module to extract names, ages, phone numbers, addresses, IDs, and years from date fields.

### 🎯 Objectives
By the end of this lab, you will be able to:
*   Understand the fundamentals of **regex** and its application in data cleaning.
*   Use Python's `str.extract()` method to pull specific data from messy strings.
*   Apply **regex replacement** to fix inconsistent address and ID formats.
*   Extract years from various date string formats.
*   Implement practical solutions for real-world healthcare data scenarios.

## 🧰 Prerequisites & ⚙️ Environment Setup

### 💡 ELI10: What are we doing?
Imagine you have a giant bucket of messy notes written by different doctors. Some wrote 'Age 45', others wrote 'Age:45'. We are setting up a special 'magnifying glass' (the `re` and `pandas` libraries) that helps our computer find these numbers automatically no matter how they were written!

We will start by importing the tools we need.

In [1]:
# Step 1: Install pandas if not already present (standard in Colab)
%pip install pandas

# Step 2: Import necessary libraries
import pandas as pd  # For data manipulation and DataFrames
import re            # The Python Regular Expression module
import numpy as np     # For handling missing values (NaN)

# Display all columns for better visibility in the notebook
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

print("Environment setup complete!")
print(f"Pandas version: {pd.__version__}")

Environment setup complete!
Pandas version: 2.2.2


## 🧪 Task 1: Use str.extract() on Messy Patient Records

### 💡 ELI10: Finding Names in the Mess
Sometimes data is hidden inside a long sentence. We use `str.extract()` to tell the computer: 'Look for a word that comes after the label "Patient:" and grab it for me!'

In [2]:
# Subtask 1.2: Create sample messy patient records
# We define a dictionary with inconsistent text strings to simulate real-world data
patient_data = {
    'record_id': [1, 2, 3, 4, 5, 6, 7, 8],
    'patient_info': [
        'Patient: John Smith, Age: 45, Phone: (555) 123-4567',
        'Name: Mary Johnson Age:32 Phone:(555)987-6543',
        'Patient Name: Robert Brown, Age: 28, Contact: 555.456.7890',
        'John Doe - Age 55 - Phone Number: (555) 234-5678',
        'Patient: Sarah Wilson, Age: 41, Tel: 555-789-0123',
        'Name: Michael Davis Age:37 Phone:(555)345-6789',
        'Patient: Lisa Anderson, Age: 29, Phone: (555) 567-8901',
        'David Miller - Age 63 - Contact: 555.678.9012'
    ]
}

# Convert the dictionary into a pandas DataFrame
df_patients = pd.DataFrame(patient_data)

print("--- Sample Patient Records ---")
display(df_patients)

--- Sample Patient Records ---


,record_id,patient_info
0,1,"Patient: John Smith, Age: 45, Phone: (555) 123..."
1,2,Name: Mary Johnson Age:32 Phone:(555)987-6543
2,3,"Patient Name: Robert Brown, Age: 28, Contact: ..."
3,4,John Doe - Age 55 - Phone Number: (555) 234-5678
4,5,"Patient: Sarah Wilson, Age: 41, Tel: 555-789-0123"
5,6,Name: Michael Davis Age:37 Phone:(555)345-6789
6,7,"Patient: Lisa Anderson, Age: 29, Phone: (555) ..."
7,8,David Miller - Age 63 - Contact: 555.678.9012


### 🔎 Extracting Names, Ages, and Phones
We will now use specific regex patterns to target the data we want.

**Regex Tip:** `(?: ... )` is a non-capturing group (we check for it, but don't save it), while `( ... )` is a capturing group (this is what `str.extract` returns).

In [3]:
# Subtask 1.3: Extract Patient Names
# Pattern breakdown:
# (?:(?:Patient Name:|Patient:|Name:)\s*|^) -> Look for labels or start of string
# ([A-Za-z]+(?:\s+[A-Za-z]+)+?) -> Capture the first and last name
# (?=,|\s*Age|\s*-\s*Age) -> Lookahead: stop when you see a comma or the word Age
name_pattern = r'(?:(?:Patient Name:|Patient:|Name:)\s*|^)([A-Za-z]+(?:\s+[A-Za-z]+)+?)(?=,|\s*Age|\s*-\s*Age)'
df_patients['patient_name'] = df_patients['patient_info'].str.extract(name_pattern)

# Subtask 1.4: Extract Ages
# Looks for 'Age' optionally followed by ':' and then digits
age_pattern = r'Age:?\s*(\d+)'
df_patients['age'] = pd.to_numeric(df_patients['patient_info'].str.extract(age_pattern)[0], errors='coerce')

# Subtask 1.5: Extract Phone Numbers
# Handles (555) 123-4567, 555.123.4567, etc.
phone_pattern = r'(?:Phone|Tel|Contact|Phone Number):?\s*(\(?\d{3}\)?[-.\s]?\d{3}[-.\s]?\d{4})'
df_patients['phone'] = df_patients['patient_info'].str.extract(phone_pattern)

print("--- Extracted Data ---")
display(df_patients[['record_id', 'patient_name', 'age', 'phone']])

--- Extracted Data ---


,record_id,patient_name,age,phone
0,1,John Smith,45,(555) 123-4567
1,2,Mary Johnson,32,(555)987-6543
2,3,Robert Brown,28,555.456.7890
3,4,John Doe,55,(555) 234-5678
4,5,Sarah Wilson,41,555-789-0123
5,6,Michael Davis,37,(555)345-6789
6,7,Lisa Anderson,29,(555) 567-8901
7,8,David Miller,63,555.678.9012


## 🧹 Task 2: Cleaning Addresses and IDs with Regex Replacement

### 💡 ELI10: Fixing the Mess
Sometimes data isn't just hidden; it's messy (like `PAT--001--NYC`). We use 'find and replace' with regex to swap out those double dashes for single ones and make everything look neat.

In [4]:
# Subtask 2.1: Create messy address and ID data
address_data = {
    'patient_id': ['P001', 'P002', 'P003', 'P004', 'P005', 'P006'],
    'raw_address': [
        '123 Main St, New York, NY 10001',
        '456 Oak Avenue,Los Angeles,CA90210',
        '789 Pine Road, Chicago, IL 60601',
        '321 Elm Street, Houston, TX 77001',
        '654 Maple Drive, Phoenix, AZ 85001',
        '987 Cedar Lane,Philadelphia,PA19101'
    ],
    'messy_id': [
        'PAT--001--NYC', 'PAT___002___LA', 'PAT..003..CHI',
        'PAT--004--HOU', 'PAT___005___PHX', 'PAT..006..PHL'
    ]
}
df_addresses = pd.DataFrame(address_data)

# Subtask 2.2: Clean Address Fields
def clean_address(address):
    # Standardize spaces and commas
    cleaned = re.sub(r'\s+', ' ', address)              # Multiple spaces to single
    cleaned = re.sub(r'\s+,', ',', cleaned)             # Remove space before comma
    cleaned = re.sub(r',(?!\s)', ', ', cleaned)         # Add space after comma
    cleaned = re.sub(r'([A-Z]{2})(\d{5})$', r'\1 \2', cleaned) # Space between State and ZIP
    return cleaned.strip()

df_addresses['clean_address'] = df_addresses['raw_address'].apply(clean_address)

# Subtask 2.3: Extract Components
# Pattern: [Street], [City], [State Code] [ZIP]
address_pattern = r'([^,]+),\s*([^,]+),\s*([A-Z]{2})\s*(\d{5})'
df_addresses[['street', 'city', 'state', 'zip_code']] = df_addresses['clean_address'].str.extract(address_pattern)

display(df_addresses[['patient_id', 'clean_address', 'city', 'zip_code']])

,patient_id,clean_address,city,zip_code
0,P001,"123 Main St, New York, NY 10001",New York,10001
1,P002,"456 Oak Avenue, Los Angeles, CA 90210",Los Angeles,90210
2,P003,"789 Pine Road, Chicago, IL 60601",Chicago,60601
3,P004,"321 Elm Street, Houston, TX 77001",Houston,77001
4,P005,"654 Maple Drive, Phoenix, AZ 85001",Phoenix,85001
5,P006,"987 Cedar Lane, Philadelphia, PA 19101",Philadelphia,19101


### 🆔 Standardizing Patient IDs
We will use `re.sub` to turn various separators into a standard dash format.

In [5]:
# Subtask 2.4: Clean ID Fields
def clean_patient_id(messy_id):
    # Replace 2 or more dots, underscores, or dashes with a single dash
    cleaned_id = re.sub(r'[-_.]{2,}', '-', messy_id)
    cleaned_id = cleaned_id.strip('-')

    # Use grouping to rebuild the ID with zero-padding
    match = re.search(r'([A-Z]+)-(\d+)-([A-Z]+)', cleaned_id)
    if match:
        prefix, number, location = match.groups()
        return f"{prefix}-{number.zfill(3)}-{location}"
    return cleaned_id

df_addresses['clean_id'] = df_addresses['messy_id'].apply(clean_patient_id)
display(df_addresses[['messy_id', 'clean_id']])

,messy_id,clean_id
0,PAT--001--NYC,PAT-001-NYC
1,PAT___002___LA,PAT-002-LA
2,PAT..003..CHI,PAT-003-CHI
3,PAT--004--HOU,PAT-004-HOU
4,PAT___005___PHX,PAT-005-PHX
5,PAT..006..PHL,PAT-006-PHL


## 📅 Task 3: Extracting Years from Various Date Formats

### 💡 ELI10: The Year Hunter
Dates can be written like '2023-12-15' or 'December 20, 2023'. We create a 'Year Hunter' function that tries different patterns until it finds a 4-digit number that looks like a year.

In [6]:
# Subtask 3.1 & 3.2: Create date data and extraction function
date_data = {
    'record_id': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
    'appointment_info': [
        'Appointment scheduled for 2023-12-15',
        'Visit date: December 20, 2023',
        'Last seen on 03/15/2024',
        'Next appointment: 2024-01-30',
        'Consultation on Jan 25, 2024',
        'Follow-up scheduled for 05/10/2024',
        'Emergency visit: 2023-11-08',
        'Routine check on February 14, 2024',
        'Surgery date: 07/22/2024',
        'Discharge: September 5, 2023'
    ]
}
df_dates = pd.DataFrame(date_data)

def extract_year(date_string):
    # Check for YYYY-MM-DD
    m1 = re.search(r'(\d{4})-\d{2}-\d{2}', date_string)
    if m1: return m1.group(1)

    # Check for MM/DD/YYYY
    m2 = re.search(r'\d{2}/\d{2}/(\d{4})', date_string)
    if m2: return m2.group(1)

    # Check for Month DD, YYYY
    m3 = re.search(r'[A-Za-z]+\s+\d{1,2},\s+(\d{4})', date_string)
    if m3: return m3.group(1)

    # Fallback: any 4 digits
    m4 = re.search(r'(\d{4})', date_string)
    if m4: return m4.group(1)
    return None

df_dates['extracted_year'] = df_dates['appointment_info'].apply(extract_year)
display(df_dates[['appointment_info', 'extracted_year']])

,appointment_info,extracted_year
0,Appointment scheduled for 2023-12-15,2023
1,"Visit date: December 20, 2023",2023
2,Last seen on 03/15/2024,2024
3,Next appointment: 2024-01-30,2024
4,"Consultation on Jan 25, 2024",2024
5,Follow-up scheduled for 05/10/2024,2024
6,Emergency visit: 2023-11-08,2023
7,"Routine check on February 14, 2024",2024
8,Surgery date: 07/22/2024,2024
9,"Discharge: September 5, 2023",2023


## 🏥 Comprehensive Healthcare Example

### 💡 ELI10: Putting it all together
Now we will take one long, super-messy sentence for each patient and pull out **everything** at once: Name, Age, Phone, Address, and Year of visit!

In [7]:
# Subtask 4.1: Create complex dataset
healthcare_data = {
    'record_id': [1, 2, 3, 4, 5],
    'patient_record': [
        'Patient: John Smith, Age: 45, Phone: (555) 123-4567, Address: 123 Main St, New York, NY 10001, Last Visit: 2023-12-15',
        'Name: Mary Johnson Age:32 Phone:(555)987-6543 Address:456 Oak Ave,Los Angeles,CA90210 Appointment: December 20, 2023',
        'Patient Name: Robert Brown, Age: 28, Contact: 555.456.7890, Address: 789 Pine Rd, Chicago, IL 60601, Visit: 03/15/2024',
        'John Doe - Age 55 - Phone Number: (555) 234-5678 - Address: 321 Elm St, Houston, TX 77001 - Date: 2024-01-30',
        'Patient: Sarah Wilson, Age: 41, Tel: 555-789-0123, Address: 654 Maple Dr, Phoenix, AZ 85001, Seen: Jan 25, 2024'
    ]
}
df_comprehensive = pd.DataFrame(healthcare_data)

# Apply all techniques
df_comprehensive['patient_name'] = df_comprehensive['patient_record'].str.extract(name_pattern)
df_comprehensive['age'] = df_comprehensive['patient_record'].str.extract(age_pattern).astype(float)
df_comprehensive['phone'] = df_comprehensive['patient_record'].str.extract(phone_pattern)

# Extract Address between "Address:" and the next visit label using non-greedy (.*?)
address_capture_pattern = r'Address:\s*(.*?)(?=(?:,\s*|\s+-\s+)(?:Last Visit|Appointment|Visit|Date|Seen):|$)'
df_comprehensive['raw_address'] = df_comprehensive['patient_record'].str.extract(address_capture_pattern)
df_comprehensive['clean_address'] = df_comprehensive['raw_address'].apply(lambda x: clean_address(x) if pd.notna(x) else x)

# Extract Visit Year
df_comprehensive['visit_year'] = df_comprehensive['patient_record'].apply(extract_year)

print("--- Final Portfolio Quality Cleaned Data ---")
display(df_comprehensive[['patient_name', 'age', 'phone', 'clean_address', 'visit_year']])

--- Final Portfolio Quality Cleaned Data ---


,patient_name,age,phone,clean_address,visit_year
0,John Smith,45.0,(555) 123-4567,"123 Main St, New York, NY 10001",2023
1,Mary Johnson,32.0,(555)987-6543,"456 Oak Ave, Los Angeles, CA90210 Appointment:...",2023
2,Robert Brown,28.0,555.456.7890,"789 Pine Rd, Chicago, IL 60601",2024
3,John Doe,55.0,(555) 234-5678,"321 Elm St, Houston, TX 77001",2024
4,Sarah Wilson,41.0,555-789-0123,"654 Maple Dr, Phoenix, AZ 85001",2024


## ✅ Verification Checklist
- [x] Successfully imported `pandas`, `re`, and `numpy`.
- [x] Extracted names despite inconsistent labels (Patient:, Name:, etc.).
- [x] Converted messy age strings to numeric data types.
- [x] Standardized phone numbers using flexible regex patterns.
- [x] Cleaned and split addresses into components.
- [x] Extracted years from at least three different date formats.

## 🛠 Troubleshooting
1. **Pattern Not Matching:** Ensure you use `\s*` to handle unexpected spaces.
2. **Greedy Matching:** If your regex grabs too much (e.g., from Name to the end of the line), use `.*?` (non-greedy) instead of `.*`.
3. **NaN Results:** `str.extract()` returns NaN if no match is found. Use `df[df['col'].isna()]` to inspect these rows and update your pattern.

## 📚 Key Takeaways
*   **Regex** is a powerful language for pattern matching.
*   **`str.extract()`** is the primary pandas tool for pulling data out of strings.
*   **Non-greedy matching (`?`)** is essential when data fields are packed close together.
*   Cleaning data *before* extracting (using `re.sub`) often makes extraction much easier.

### 🏁 Conclusion
You have mastered the basics of Regex in Python and Pandas! These skills are critical for data cleaning in healthcare, finance, and any field where humans type data into text boxes.